In [1]:
# Import libraries used for data loading, manipulation, and visualization.
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Load the raw hospital patient dataset into a Pandas DataFrame.
df = pd.read_csv("Hospital Patient.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Hospital Patient.csv'

In [ ]:
# Display the raw dataset to understand its structure and contents.
df

In [ ]:
# Check the number of rows and columns in the raw dataset.
print("Dataset Shape:", df.shape)

In [ ]:
# Review the column names to understand the available fields.
print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# Inspect the DataFrame structure and data types.
print("\nDataset Information")
print(df.info)

In [ ]:
# Check every column for missing values before analysis.
print("\nMissing Information")
print(df.isnull().sum())

In [ ]:
# Check for exact duplicate records in the raw dataset.
print("\nDuplicate Rows:",df.duplicated().sum())

In [ ]:
# Review descriptive statistics for the main numeric variables.
print("\nSummary Statistics")
print(df.describe())

In [ ]:
# Check whether Patient_ID is unique
df['Patient_ID'].is_unique

In [ ]:
# Check for duplicate Patient IDs
df['Patient_ID'].duplicated().sum()

In [ ]:
# Inspect the data type of each column.
df.dtypes

In [ ]:
# Review the unique categories in the main categorical fields.
categorical_columns = [
    'Gender',
    'Condition',
    'Medication ',
    'Patient_State',
    'Readmission',
    'Outcome',
    'Insurance_Claimed'
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].unique())

In [ ]:
# Inspect the raw date values and confirm they are initially stored as strings.
print(df['Admission_Date'].head())
print(df['Discharge_Date'].head())

print("\nData types:")
print(df[['Admission_Date', 'Discharge_Date']].dtypes)

In [ ]:
# Convert admission and discharge dates from strings to datetime values.
admission_dates = pd.to_datetime(
    df['Admission_Date'],
    format='%d-%m-%Y'
)

discharge_dates = pd.to_datetime(
    df['Discharge_Date'],
    format='%d-%m-%Y'
)

In [ ]:
# Check for impossible records where discharge occurs before admission.
(discharge_dates < admission_dates).sum()

In [ ]:
# Calculate length of stay directly from the admission and discharge dates.
calculated_stay = (
    discharge_dates - admission_dates
).dt.days

In [ ]:
# Compare the recorded length of stay with the value calculated from the dates.
comparison = pd.DataFrame({
    'Recorded_Stay': df['Length_of_Stay'],
    'Calculated_Stay': calculated_stay
})

comparison['Difference'] = (
    comparison['Recorded_Stay'] -
    comparison['Calculated_Stay']
)

comparison.head(10)

In [ ]:
# Examine the differences between recorded and calculated length of stay.
comparison['Difference'].value_counts().sort_index()

In [ ]:
# Count records where the recorded and calculated stays do not match.
(comparison['Difference'] != 0).sum()

In [ ]:
# Inspect the range and basic characteristics of the main numeric fields.
numeric_columns = [
    'Age',
    'Year_of_Admission',
    'Length_of_Stay',
    'Satisfaction',
    'Total_Cost'
]

for column in numeric_columns:
    print(f"\n{column}")
    print("Minimum:", df[column].min())
    print("Maximum:", df[column].max())
    print("Unique values:", df[column].nunique())

In [ ]:
# Check for ages outside a reasonable human age range.
print("Age values outside expected range:")
print(df[(df['Age'] < 0) | (df['Age'] > 100)])

In [ ]:
# Confirm that satisfaction scores use the expected 2-to-5 scale.
print("Satisfaction values:")
print(sorted(df['Satisfaction'].unique()))

In [ ]:
# Confirm the valid Readmission categories.
print("Readmission values:")
print(df['Readmission'].unique())

In [ ]:
# Extract the admission year from the converted admission dates.
calculated_year = admission_dates.dt.year

In [ ]:
# Verify that the recorded admission year matches the year in Admission_Date.
year_check = pd.DataFrame({
    'Recorded_Year': df['Year_of_Admission'],
    'Calculated_Year': calculated_year
})

year_check['Match'] = (
    year_check['Recorded_Year'] ==
    year_check['Calculated_Year']
)

year_check['Match'].value_counts()

In [ ]:
# Review the gender and age distributions before creating age segments.
print("Gender Distribution:")
print(df['Gender'].value_counts())

print("\nAge Distribution:")
print(df['Age'].value_counts().sort_index())

In [ ]:
# Group patients into age bands for segment-level analysis.
age_groups = pd.cut(
    df['Age'],
    bins=[24, 34, 44, 54, 64, 74, 100],
    labels=['25-34', '35-44', '45-54', '55-64', '65-74', '75+']
)


In [ ]:
# Count patients by medical condition.
condition_counts = df['Condition'].value_counts()

print(condition_counts)

In [ ]:
# Compare treatment cost across conditions using patient count, average cost, and total cost.
print(
    df.groupby('Condition')['Total_Cost']
      .agg(['count', 'mean', 'sum'])
      .sort_values('sum', ascending=False)
)

In [ ]:
# Review the overall distribution of hospital length of stay.
print(
    df['Length_of_Stay'].describe()
)

In [ ]:
# Compare hospital utilization across conditions using length-of-stay statistics.
print(
    df.groupby('Condition')['Length_of_Stay']
      .agg(['count', 'mean', 'median', 'max'])
      .sort_values('mean', ascending=False)
)

In [ ]:
# Measure the overall number and percentage of readmitted patients.
print(df['Readmission'].value_counts())

print("\nReadmission Percentage:")
print(
    df['Readmission']
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

In [ ]:
# Calculate the readmission rate for each medical condition.
readmission_by_condition = (
    df.groupby('Condition')['Readmission']
      .apply(lambda x: (x == 'Yes').mean() * 100)
      .sort_values(ascending=False)
)

print(readmission_by_condition)

In [ ]:
# Review the distribution of patient satisfaction scores.
print(df['Satisfaction'].value_counts().sort_index())

In [ ]:
# Examine treatment cost across different satisfaction levels.
print(
    df.groupby('Satisfaction')['Total_Cost']
      .agg(['count', 'mean'])
)

In [ ]:
# Examine length of stay across different satisfaction levels.
print(
    df.groupby('Satisfaction')['Length_of_Stay']
      .agg(['count', 'mean'])
)

In [ ]:
# Review overall patient outcomes and their percentages.
print(df['Outcome'].value_counts())

print("\nOutcome Percentage:")
print(
    df['Outcome']
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

In [ ]:
# Compare outcome patterns across medical conditions.
print(
    pd.crosstab(
        df['Condition'],
        df['Outcome'],
        normalize='index'
    ).mul(100).round(2)
)

In [ ]:
# Review insurance claim status and its overall distribution.
print(df['Insurance_Claimed'].value_counts())

print("\nInsurance Claim Percentage:")
print(
    df['Insurance_Claimed']
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

In [ ]:
# Compare treatment costs between patients who claimed insurance and those who did not.
print(
    df.groupby('Insurance_Claimed')['Total_Cost']
      .agg(['count', 'mean', 'sum'])
)

In [ ]:
# Calculate correlations among key numeric variables to identify useful associations for further analysis.
numeric_analysis = [
    'Age',
    'Length_of_Stay',
    'Satisfaction',
    'Total_Cost'
]

correlation_matrix = df[numeric_analysis].corr()

correlation_matrix.round(2)

# DATA CLEANING

The raw dataset is copied and standardized before creating analytical features.


In [ ]:
# Create a separate copy so the raw DataFrame remains unchanged.
clean_df = df.copy()

In [ ]:
# Remove accidental leading/trailing whitespace from column names.
clean_df.columns = clean_df.columns.str.strip()

In [ ]:
# Confirm the cleaned column names.
clean_df.columns.tolist()

In [ ]:
# Standardize text fields by removing leading/trailing whitespace from string values.
string_columns = clean_df.select_dtypes(include='str').columns

for column in string_columns:
    clean_df[column] = clean_df[column].str.strip()

In [ ]:
# Recheck the number of unique states after text cleaning.
clean_df['Patient_State'].nunique()


In [ ]:
# Confirm the standardized state names after removing whitespace.
clean_df['Patient_State'].unique()

In [ ]:
# Convert date fields to datetime format for reliable date-based analysis.
clean_df['Admission_Date'] = pd.to_datetime(
    clean_df['Admission_Date'],
    format='%d-%m-%Y'
)

clean_df['Discharge_Date'] = pd.to_datetime(
    clean_df['Discharge_Date'],
    format='%d-%m-%Y'
)

In [ ]:
# Verify that both date columns are now stored as datetime values.
clean_df[['Admission_Date', 'Discharge_Date']].dtypes

In [ ]:
# Create age groups to support demographic segmentation in the analysis and dashboard.
clean_df['Age_Group'] = pd.cut(
    clean_df['Age'],
    bins=[24, 34, 44, 54, 64, 74, 100],
    labels=['25-34', '35-44', '45-54', '55-64', '65-74', '75+']
)

In [ ]:
# Preview the new age-group classification.
clean_df[['Age', 'Age_Group']].head(10)

In [ ]:
# Create treatment-cost bands to identify low-, medium-, high-, and very-high-cost patients.
clean_df['Cost_Band'] = pd.cut(
    clean_df['Total_Cost'],
    bins=[0, 5000, 10000, 20000, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High']
)

In [ ]:
# Check the number of patients in each treatment-cost band.
clean_df['Cost_Band'].value_counts()

In [ ]:
# Convert the Yes/No readmission field into a numeric flag for calculations.
clean_df['Readmission_Flag'] = (
    clean_df['Readmission'] == 'Yes'
).astype(int)

In [ ]:
# Validate the readmission flag counts.
clean_df['Readmission_Flag'].value_counts()

In [ ]:
# Convert insurance claim status into a numeric flag for BI calculations.
clean_df['Insurance_Claim_Flag'] = (
    clean_df['Insurance_Claimed'] == 'Yes'
).astype(int)

In [ ]:
# Validate the cleaned dataset after transformation.
print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate rows:", clean_df.duplicated().sum())

print("\nUnique patients:", clean_df['Patient_ID'].nunique())

print("\nDate range:")
print(clean_df['Admission_Date'].min())
print(clean_df['Admission_Date'].max())

# ANALYSIS

The cleaned dataset is analyzed to identify cost drivers, resource utilization, readmission patterns, patient experience, and priority patient segments.


In [ ]:
# Calculate the core hospital KPIs used to frame the analysis and Power BI dashboard.
total_patients = clean_df['Patient_ID'].nunique()
total_cost = clean_df['Total_Cost'].sum()
avg_cost = clean_df['Total_Cost'].mean()
avg_length_of_stay = clean_df['Length_of_Stay'].mean()
readmission_rate = clean_df['Readmission_Flag'].mean() * 100
avg_satisfaction = clean_df['Satisfaction'].mean()


In [ ]:
# Display the hospital-level performance summary.

print("Hospital Performance Summary")
print("--------------------------------")
print(f"Total Patients: {total_patients:,}")
print(f"Total Treatment Cost: {total_cost:,.2f}")
print(f"Average Treatment Cost: {avg_cost:,.2f}")
print(f"Average Length of Stay: {avg_length_of_stay:.2f} days")
print(f"Readmission Rate: {readmission_rate:.2f}%")
print(f"Average Satisfaction: {avg_satisfaction:.2f}/5")

In [ ]:
# Aggregate treatment cost by condition to identify the largest cost drivers.
condition_cost = (
    clean_df.groupby('Condition')
    .agg(
        Patients=('Patient_ID', 'count'),
        Total_Cost=('Total_Cost', 'sum'),
        Average_Cost=('Total_Cost', 'mean')
    )
    .sort_values('Total_Cost', ascending=False)
)

condition_cost

In [ ]:
# Calculate each condition's percentage share of total treatment cost.
condition_cost['Cost_Share_%'] = (
    condition_cost['Total_Cost'] / total_cost * 100
).round(2)

condition_cost

In [ ]:
# Aggregate hospital days by condition to measure resource utilization.
condition_resources = (
    clean_df.groupby('Condition')
    .agg(
        Patients=('Patient_ID', 'count'),
        Total_Hospital_Days=('Length_of_Stay', 'sum'),
        Average_Length_of_Stay=('Length_of_Stay', 'mean')
    )
    .sort_values('Total_Hospital_Days', ascending=False)
)

condition_resources

In [ ]:
# Calculate each condition's share of total hospital days.
condition_resources['Share_of_Hospital_Days_%'] = (
    condition_resources['Total_Hospital_Days']
    / clean_df['Length_of_Stay'].sum()
    * 100
).round(2)

condition_resources

In [ ]:
# Combine cost and resource-utilization metrics into one condition-level performance table.
condition_performance = condition_cost.merge(
    condition_resources,
    on='Condition'
)

condition_performance

In [ ]:
# Sort conditions by total treatment cost for easier interpretation.
condition_performance.sort_values(
    'Total_Cost',
    ascending=False
)

In [ ]:
# Calculate readmissions and readmission rates by medical condition.
readmission_analysis = (
    clean_df.groupby('Condition')
    .agg(
        Patients=('Patient_ID', 'count'),
        Readmissions=('Readmission_Flag', 'sum'),
        Readmission_Rate=('Readmission_Flag', 'mean')
    )
)

readmission_analysis['Readmission_Rate'] = (
    readmission_analysis['Readmission_Rate'] * 100
).round(2)

readmission_analysis = readmission_analysis.sort_values(
    'Readmission_Rate',
    ascending=False
)

readmission_analysis

In [ ]:
# Compare readmitted and non-readmitted patients on satisfaction, stay, and average cost.
readmission_satisfaction = (
    clean_df.groupby('Readmission')
    .agg(
        Patients=('Patient_ID', 'count'),
        Average_Satisfaction=('Satisfaction', 'mean'),
        Average_Length_of_Stay=('Length_of_Stay', 'mean'),
        Average_Cost=('Total_Cost', 'mean')
    )
)

readmission_satisfaction.round(2)

In [ ]:
# Compare cost, length of stay, readmission, and satisfaction across age groups.
age_analysis = (
    clean_df.groupby('Age_Group', observed=True)
    .agg(
        Patients=('Patient_ID', 'count'),
        Average_Cost=('Total_Cost', 'mean'),
        Total_Cost=('Total_Cost', 'sum'),
        Average_Length_of_Stay=('Length_of_Stay', 'mean'),
        Readmission_Rate=('Readmission_Flag', 'mean'),
        Average_Satisfaction=('Satisfaction', 'mean')
    )
)

age_analysis['Readmission_Rate'] = (
    age_analysis['Readmission_Rate'] * 100
).round(2)

age_analysis.round(2)

In [ ]:
# Identify patients who combine relatively high treatment cost with low satisfaction.
high_cost_low_satisfaction = clean_df[
    (clean_df['Cost_Band'].isin(['High', 'Very High'])) &
    (clean_df['Satisfaction'] <= 3)
]

print("Patients with high cost and low satisfaction:",
      len(high_cost_low_satisfaction))

print("\nPercentage of all patients:",
      round(len(high_cost_low_satisfaction) / total_patients * 100, 2), "%")

In [ ]:
# Identify which conditions contribute most to the high-cost, low-satisfaction segment.
high_cost_low_satisfaction.groupby('Condition').agg(
    Patients=('Patient_ID', 'count'),
    Average_Cost=('Total_Cost', 'mean'),
    Average_Length_of_Stay=('Length_of_Stay', 'mean'),
    Average_Satisfaction=('Satisfaction', 'mean')
).sort_values(
    'Patients',
    ascending=False
)

In [ ]:
# Compare patient experience and readmission patterns across treatment-cost bands.
cost_satisfaction = (
    clean_df.groupby('Cost_Band', observed=True)
    .agg(
        Patients=('Patient_ID', 'count'),
        Average_Cost=('Total_Cost', 'mean'),
        Average_Satisfaction=('Satisfaction', 'mean'),
        Average_Length_of_Stay=('Length_of_Stay', 'mean'),
        Readmission_Rate=('Readmission_Flag', 'mean')
    )
)

cost_satisfaction['Readmission_Rate'] = (
    cost_satisfaction['Readmission_Rate'] * 100
).round(2)

cost_satisfaction.round(2)

In [ ]:
# Compare hospital performance metrics across patient states.
state_analysis = (
    clean_df.groupby('Patient_State')
    .agg(
        Patients=('Patient_ID', 'count'),
        Total_Cost=('Total_Cost', 'sum'),
        Average_Cost=('Total_Cost', 'mean'),
        Average_Length_of_Stay=('Length_of_Stay', 'mean'),
        Readmission_Rate=('Readmission_Flag', 'mean'),
        Average_Satisfaction=('Satisfaction', 'mean')
    )
)

state_analysis['Readmission_Rate'] = (
    state_analysis['Readmission_Rate'] * 100
).round(2)

state_analysis.round(2)

In [ ]:
# Export the cleaned dataset for use in Power BI and for reproducibility.
clean_df.to_csv('healthcare_clean.csv', index=False)

print('Cleaned dataset exported successfully.')
print('Shape:', clean_df.shape)
